In [ ]:
# ── Config ──────────────────────────────────────────────────
SWC_FILE   = 'output/neurons_riem.swc'
STACK_TIF  = 'output/stack_preprocessed.tif'
SOMA_NPZ   = 'output/soma.npz'
PREP_NPZ   = 'output/prep_riem.npz'

SLICE_THICK_VOX = 3    # 2D overlay: ±N voxel 이내 노드 표시
DS3D            = 4    # 3D 슬라이스용 다운샘플 배율 (메모리 절약)

In [ ]:
# ── Load ────────────────────────────────────────────────────
import numpy as np
import tifffile
from collections import defaultdict

# SWC
swc_nodes = {}
with open(SWC_FILE) as f:
    for line in f:
        if line.startswith('#') or not line.strip(): continue
        p = line.split()
        nid = int(p[0])
        swc_nodes[nid] = dict(
            type=int(p[1]), x=float(p[2]), y=float(p[3]),
            z=float(p[4]), r=float(p[5]), parent=int(p[6]))

# Soma mesh
soma_data   = np.load(SOMA_NPZ)
sv_local    = soma_data['mesh_verts'].astype(np.float64)
sf          = soma_data['mesh_faces']
soma_cv     = soma_data['soma_centroid_vox']
voxel_iso_s = float(soma_data['voxel_iso'])
sv = sv_local + (soma_cv * voxel_iso_s - sv_local.mean(axis=0))

# voxel_iso
voxel_iso = float(np.load(PREP_NPZ)['voxel_iso'])

# Raw stack
print('Loading stack...', flush=True)
stack = tifffile.imread(STACK_TIF).astype(np.float32)
NZ, NY, NX = stack.shape
stack_max = float(stack.max())
print(f'Stack: {stack.shape}  voxel={voxel_iso:.4f} µm  max={stack_max:.0f}')

# children / branch order
children_map = defaultdict(list)
for nid, n in swc_nodes.items():
    if n['parent'] != -1:
        children_map[n['parent']].append(nid)

branch_order = {1: 0}
queue = [1]
while queue:
    cur  = queue.pop(0)
    kids = children_map.get(cur, [])
    for ch in kids:
        branch_order[ch] = branch_order[cur] + (1 if len(kids) >= 2 else 0)
        queue.append(ch)

# Soma center in voxel coords
soma_z_px = swc_nodes[1]['z'] / voxel_iso
soma_y_px = swc_nodes[1]['y'] / voxel_iso
soma_x_px = swc_nodes[1]['x'] / voxel_iso

ORDER_COLORS = {0:'#888',1:'#FF4444',2:'#FF8C00',3:'#FFD700',4:'#44BB44',5:'#4488FF'}
def order_color(o): return ORDER_COLORS.get(min(o,5),'#88AAFF')

print('Load complete.')

In [ ]:
# ── 3D: Raw slice planes + Neuron mesh ──────────────────────
import plotly.graph_objects as go
from scipy.ndimage import zoom

# 3D용 다운샘플
ds = DS3D
st = stack[::ds, ::ds, ::ds]
dz, dy, dx = st.shape
vi_ds = voxel_iso * ds   # µm/voxel after downsample

z0 = int(soma_z_px / ds)
y0 = int(soma_y_px / ds)
x0 = int(soma_x_px / ds)
z0 = np.clip(z0, 0, dz-1)
y0 = np.clip(y0, 0, dy-1)
x0 = np.clip(x0, 0, dx-1)

Xg = np.arange(dx) * vi_ds
Yg = np.arange(dy) * vi_ds
Zg = np.arange(dz) * vi_ds

def norm_slice(s):
    s = s / (stack_max * 0.7)
    return np.clip(s, 0, 1)

fig = go.Figure()

# XY plane (Z=soma)
XX, YY = np.meshgrid(Xg, Yg)
fig.add_trace(go.Surface(
    x=XX, y=YY, z=np.full_like(XX, z0*vi_ds),
    surfacecolor=norm_slice(st[z0]),
    colorscale='Gray', showscale=False, opacity=0.85,
    name=f'XY z={z0*vi_ds:.0f}µm',
))

# XZ plane (Y=soma)
XX2, ZZ2 = np.meshgrid(Xg, Zg)
fig.add_trace(go.Surface(
    x=XX2, y=np.full_like(XX2, y0*vi_ds), z=ZZ2,
    surfacecolor=norm_slice(st[:, y0, :]),
    colorscale='Gray', showscale=False, opacity=0.7,
    name=f'XZ y={y0*vi_ds:.0f}µm',
))

# YZ plane (X=soma)
YY3, ZZ3 = np.meshgrid(Yg, Zg)
fig.add_trace(go.Surface(
    x=np.full_like(YY3, x0*vi_ds), y=YY3, z=ZZ3,
    surfacecolor=norm_slice(st[:, :, x0]),
    colorscale='Gray', showscale=False, opacity=0.7,
    name=f'YZ x={x0*vi_ds:.0f}µm',
))

# Soma mesh
fig.add_trace(go.Mesh3d(
    x=sv[:,2], y=sv[:,1], z=sv[:,0],
    i=sf[:,0], j=sf[:,1], k=sf[:,2],
    color='#FF6B6B', opacity=0.7, name='Soma',
    lighting=dict(ambient=0.5, diffuse=0.6),
))

# Neuron branches (Scatter3d lines, branch order 색상)
segs = defaultdict(lambda: dict(x=[], y=[], z=[]))
for nid, n in swc_nodes.items():
    par = n['parent']
    if par == -1 or par not in swc_nodes: continue
    p = swc_nodes[par]
    o = branch_order.get(nid, 0)
    segs[o]['x'] += [n['x'], p['x'], None]
    segs[o]['y'] += [n['y'], p['y'], None]
    segs[o]['z'] += [n['z'], p['z'], None]

OL = {1:'1차',2:'2차',3:'3차',4:'4차'}
for o in sorted(segs.keys()):
    if o == 0: continue
    fig.add_trace(go.Scatter3d(
        **segs[o], mode='lines',
        line=dict(color=order_color(o), width=2 if o<=2 else 1),
        name=OL.get(o, f'{o}차'), opacity=0.95,
    ))

fig.update_layout(
    title=dict(text='3D: Raw slices (soma center) + Neuron mesh', font=dict(color='white')),
    scene=dict(
        xaxis_title='X (µm)', yaxis_title='Y (µm)', zaxis_title='Z (µm)',
        bgcolor='#050505',
        xaxis=dict(backgroundcolor='#050505', gridcolor='#1a1a1a', color='white'),
        yaxis=dict(backgroundcolor='#050505', gridcolor='#1a1a1a', color='white'),
        zaxis=dict(backgroundcolor='#050505', gridcolor='#1a1a1a', color='white'),
        aspectmode='data',
    ),
    paper_bgcolor='#111111', font_color='white',
    margin=dict(l=0, r=0, t=50, b=0),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(20,20,20,0.85)'),
)
fig.show()
del st

In [ ]:
# ── 2D 슬라이스 스크롤: Raw + SWC overlay ───────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

# SWC 노드를 voxel 좌표로 변환해 캐시
swc_px = {nid: (
    n['z'] / voxel_iso,   # z_px
    n['y'] / voxel_iso,   # y_px
    n['x'] / voxel_iso,   # x_px
    branch_order.get(nid, 0),
    n['parent']
) for nid, n in swc_nodes.items()}

VMAX = stack_max * 0.75

def show_z(z=int(soma_z_px)):
    z = int(np.clip(z, 0, NZ-1))
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
    fig.patch.set_facecolor('#111111')

    for ax in axes:
        ax.imshow(stack[z], cmap='gray', origin='upper',
                  vmin=0, vmax=VMAX, aspect='equal')
        ax.axis('off')

    axes[0].set_title(f'Raw  Z={z}  ({z*voxel_iso:.1f} µm)', color='white', fontsize=11)
    axes[1].set_title(f'SWC overlay  Z={z}  (±{SLICE_THICK_VOX} vox)', color='white', fontsize=11)

    # SWC 세그먼트 그리기
    for nid, (nz, ny, nx, o, par) in swc_px.items():
        if par == -1 or par not in swc_px: continue
        pz, py, px, _, _ = swc_px[par]
        # 현재 Z 슬라이스에 걸치는 세그먼트만
        z_min = min(nz, pz); z_max = max(nz, pz)
        if z_max < z - SLICE_THICK_VOX or z_min > z + SLICE_THICK_VOX:
            continue
        alpha = max(0.3, 1.0 - abs((nz+pz)/2 - z) / (SLICE_THICK_VOX + 1))
        axes[1].plot([nx, px], [ny, py], '-',
                     color=order_color(o), lw=0.8, alpha=alpha)

    # soma 표시
    if abs(soma_z_px - z) < SLICE_THICK_VOX * 2:
        axes[1].scatter([soma_x_px], [soma_y_px],
                        c='white', s=120, marker='*', zorder=5, linewidths=0)

    legend = [mpatches.Patch(color=c, label=l)
              for c, l in [('#FF4444','1차'),('#FF8C00','2차'),
                           ('#FFD700','3차'),('#44BB44','4차'),('#4488FF','5차+')]]
    axes[1].legend(handles=legend, facecolor='#222', labelcolor='white',
                   fontsize=8, loc='upper right')
    plt.show()

z_slider = widgets.IntSlider(
    value=int(soma_z_px), min=0, max=NZ-1, step=1,
    description='Z slice:', continuous_update=False,
    layout=widgets.Layout(width='700px'),
    style={'description_width': '80px'},
)
widgets.interact(show_z, z=z_slider);

In [ ]:
# ── 2D 슬라이스 스크롤: XZ / YZ 단면 ───────────────────────
import ipywidgets as widgets

def show_xz(y=int(soma_y_px)):
    y = int(np.clip(y, 0, NY-1))
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
    fig.patch.set_facecolor('#111111')
    for ax in axes:
        ax.imshow(stack[:, y, :], cmap='gray', origin='upper',
                  vmin=0, vmax=VMAX, aspect='equal')
        ax.axis('off')
    axes[0].set_title(f'Raw XZ  Y={y}  ({y*voxel_iso:.1f} µm)', color='white')
    axes[1].set_title(f'SWC overlay XZ  Y={y}', color='white')
    for nid, (nz, ny, nx, o, par) in swc_px.items():
        if par == -1 or par not in swc_px: continue
        pz, py, px, _, _ = swc_px[par]
        y_mid = (ny + py) / 2
        if abs(y_mid - y) > SLICE_THICK_VOX: continue
        alpha = max(0.3, 1.0 - abs(y_mid - y) / (SLICE_THICK_VOX + 1))
        axes[1].plot([nx, px], [nz, pz], '-',
                     color=order_color(o), lw=0.8, alpha=alpha)
    plt.show()

def show_yz(x=int(soma_x_px)):
    x = int(np.clip(x, 0, NX-1))
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
    fig.patch.set_facecolor('#111111')
    for ax in axes:
        ax.imshow(stack[:, :, x], cmap='gray', origin='upper',
                  vmin=0, vmax=VMAX, aspect='equal')
        ax.axis('off')
    axes[0].set_title(f'Raw YZ  X={x}  ({x*voxel_iso:.1f} µm)', color='white')
    axes[1].set_title(f'SWC overlay YZ  X={x}', color='white')
    for nid, (nz, ny, nx_, o, par) in swc_px.items():
        if par == -1 or par not in swc_px: continue
        pz, py, px, _, _ = swc_px[par]
        x_mid = (nx_ + px) / 2
        if abs(x_mid - x) > SLICE_THICK_VOX: continue
        alpha = max(0.3, 1.0 - abs(x_mid - x) / (SLICE_THICK_VOX + 1))
        axes[1].plot([ny, py], [nz, pz], '-',
                     color=order_color(o), lw=0.8, alpha=alpha)
    plt.show()

print('=== XZ 단면 (Y 방향 스크롤) ===')
y_slider = widgets.IntSlider(
    value=int(soma_y_px), min=0, max=NY-1, step=1,
    description='Y slice:', continuous_update=False,
    layout=widgets.Layout(width='700px'),
    style={'description_width': '80px'},
)
widgets.interact(show_xz, y=y_slider);

print('=== YZ 단면 (X 방향 스크롤) ===')
x_slider = widgets.IntSlider(
    value=int(soma_x_px), min=0, max=NX-1, step=1,
    description='X slice:', continuous_update=False,
    layout=widgets.Layout(width='700px'),
    style={'description_width': '80px'},
)
widgets.interact(show_yz, x=x_slider);